In [ ]:
import threading
import time
import tkinter as tk
from tkinter import ttk


class TimeInterval:

    def __init__(self, hours=0, minutes=0, seconds=0):
        self.hours = hours
        self.minutes = minutes
        self.seconds = seconds

    @staticmethod
    def __is_float_or_int(other: object) -> bool:
        return type(other) is int

    @property
    def hours(self):
        return self.__hours

    @hours.setter
    def hours(self, other):
        if TimeInterval.__is_float_or_int(other):
            if other < 0 or other >= 24:
                raise ValueError('Часы не могут быть отрицательными или больше 24')
            else:
                self.__hours = other
        else:
            raise TypeError('Время должно быть числом')

    @hours.deleter
    def hours(self):
        del self.__hours

    @property
    def minutes(self):
        return self.__minutes

    @minutes.setter
    def minutes(self, other):
        if TimeInterval.__is_float_or_int(other):
            if other < 0:
                raise ValueError('Минуты не могут быть отрицательными')
            elif other >= 60:
                self.__hours = (self.__hours + other // 60) % 24
                self.__minutes = other % 60
            else:
                self.__minutes = other
        else:
            raise TypeError('Время должно быть числом')

    @minutes.deleter
    def minutes(self):
        del self.__minutes

    @property
    def seconds(self):
        return self.__seconds

    @seconds.setter
    def seconds(self, other):
        if TimeInterval.__is_float_or_int(other):
            if other < 0:
                raise ValueError('Секунды не могут быть отрицательными')
            else:
                total_seconds = self.__hours * 3600 + self.__minutes * 60 + other  # Прибавляем переданное значение к текущему времени
                total_seconds %= 86400  # Нормализуем в пределах 0–86399 секунд (сутки)
                self.__hours = total_seconds // 3600
                self.__minutes = (total_seconds % 3600) // 60
                self.__seconds = total_seconds % 60
        else:
            raise TypeError('Время должно быть числом')

    @seconds.deleter
    def seconds(self):
        del self.__seconds

    def __str__(self):
        return f'{self.hours:02}:{self.minutes:02}:{self.seconds:02}'

    def __repr__(self):
        return f'TimeInterval({self.hours}, {self.minutes}, {self.seconds})'

    @staticmethod
    def _to_seconds(other: object):
        if isinstance(other, TimeInterval):
            return other._total_seconds()
        elif isinstance(other, str):
            if TimeInterval.is_valid(other):
                hours, minutes, seconds = map(int, other.split(':'))
                return hours * 3600 + minutes * 60 + seconds
            else:
                raise ValueError('Не поддерживаемый формат строки для времени')
        else:
            raise TypeError('Неподдерживаемый тип')

    def _total_seconds(self):
        return self.__hours * 3600 + self.__minutes * 60 + self.__seconds

    @staticmethod
    def _converter_seconds(total_sec):
        hours = total_sec // 3600
        minutes = (total_sec % 3600) // 60
        seconds = total_sec % 60
        return hours, minutes, seconds

    def __add__(self, other):
        if isinstance(other, TimeInterval) or TimeInterval.is_valid(other):
            total_sec = self._total_seconds() + TimeInterval._to_seconds(other)
            total_sec %= 86400  # сутки
            hours, minutes, seconds = TimeInterval._converter_seconds(total_sec)
            return TimeInterval(hours, minutes, seconds)
        else:
            raise ValueError('Неправильная строка времени')

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        if TimeInterval.is_valid(other):
            total_self_sec = self._total_seconds()
            total_other_sec = TimeInterval._to_seconds(other)
            total = total_self_sec - total_other_sec
            if total <= 0:
                print('Время обнулилось')
                return TimeInterval(0, 0, 0)
            else:
                hours, minutes, seconds = TimeInterval._converter_seconds(total)
                return TimeInterval(hours, minutes, seconds)
        else:
            raise ValueError('Неправильная строка времени')

    def __rsub__(self, other):
        return self - other

    def __mul__(self, other):
        if type(other) is int:  # потому что если isinstance(), а other будет True/ False, то пройдет проверку
            total_sec = self._total_seconds() * other
            total_sec %= 86400  # нормализуем в сутки
            hours, minutes, seconds = TimeInterval._converter_seconds(total_sec)
            return TimeInterval(hours, minutes, seconds)
        else:
            raise ValueError(F'Нельзя умножить время на {other}')

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        if isinstance(other, TimeInterval):
            total_self_sec = self._total_seconds()
            total_other_sec = other._total_seconds()
            div = total_self_sec / total_other_sec // 1
            hours, minutes, seconds = TimeInterval._converter_seconds(div)
            return TimeInterval(hours, minutes, seconds)
        elif type(other) in (int, float):
            if other == 0:
                raise ValueError('Нельзя делить на ноль')
            else:
                total_self_sec = self._total_seconds()
                div = total_self_sec / other // 1
                hours, minutes, seconds = TimeInterval._converter_seconds(div)
                return TimeInterval(hours, minutes, seconds)

        else:
            raise ValueError(f'Не правильный тип данных для деления времени  на {other}')

    def __rtruediv__(self, other):
        return self / other

    def __eq__(self, other):
        total_self_sec = self._total_seconds()
        total_other_sec = TimeInterval._to_seconds(other)
        return total_self_sec == total_other_sec

    def __lt__(self, other):
        total_self_sec = self._total_seconds()
        total_other_sec = TimeInterval._to_seconds(other)
        return total_self_sec < total_other_sec

    def __le__(self, other):
        return self == other or self < other

    def __gt__(self, other):
        return not (self <= other)

    def __ge__(self, other):
        return not (self < other)

    def __len__(self):
        return self._total_seconds()

    def __bool__(self):
        return len(self) != 0

    @classmethod
    def from_string(cls, other):
        res = cls._to_seconds(other)
        hours, minutes, seconds = TimeInterval._converter_seconds(res)
        return TimeInterval(hours, minutes, seconds)

    @staticmethod
    def is_valid(other: str):
        if isinstance(other, TimeInterval):
            return True
        elif isinstance(other, str):
            if len(other) == 8 and (other[2] == ':' and other[5] == ':') and (
                    other[0:2].isdigit() and other[3:5].isdigit() and other[6:].isdigit()):
                return True
            else:
                return False
        else:
            return False


class Timer(TimeInterval):
    def __init__(self, hours=0, minutes=0, seconds=0):
        super().__init__(hours, minutes, seconds)
        self._initial_seconds = self._total_seconds()
        self._running = False
        self._stop_flag = False

    def _set_time(self, seconds):
        """Устанавливает время через свойства родителя."""
        h = seconds // 3600
        m = (seconds % 3600) // 60
        s = seconds % 60
        self.hours = h
        self.minutes = m
        self.seconds = s

    def start(self, callback=None):
        if self._total_seconds() <= 0:
            print("Таймер уже на нуле")
            return
        self._running = True
        self._stop_flag = False
        end_time = time.time() + self._total_seconds()
        print(f'Запускаю таймер на {str(self)}')
        while time.time() < end_time and not self._stop_flag:
            remaining = int(end_time - time.time())
            self._set_time(remaining)
            if callback:
                callback()
            print(f'\rОсталось {self}', end='')
            time.sleep(max(0, 1 - (time.time() % 1) + 0.001))
        if self._stop_flag:
            print("\nТаймер остановлен")
        else:
            print("\nТаймер завершён!")
        self._running = False
        if callback:
            callback()

    def stop(self):
        self._stop_flag = True
        self._running = False

    def reset(self):
        self._stop_flag = True
        self._running = False
        self._set_time(self._initial_seconds)


class TimerGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Таймер")

        # Поле для ввода времени (в секундах)
        ttk.Label(root, text="Введите время (сек):").pack(pady=5)
        self.entry = ttk.Entry(root)
        self.entry.pack(pady=5)

        # Кнопка установки
        self.set_btn = ttk.Button(root, text="Установить", command=self.set_timer)
        self.set_btn.pack(pady=5)

        # Метка для отображения времени
        self.time_label = ttk.Label(root, text="00:00:00", font=("Arial", 24))
        self.time_label.pack(pady=10)

        # Кнопки управления
        self.start_btn = ttk.Button(root, text="Старт", command=self.start_timer)
        self.start_btn.pack(side=tk.LEFT, padx=5)
        self.stop_btn = ttk.Button(root, text="Стоп", command=self.stop_timer)
        self.stop_btn.pack(side=tk.LEFT, padx=5)
        self.reset_btn = ttk.Button(root, text="Сброс", command=self.reset_timer)
        self.reset_btn.pack(side=tk.LEFT, padx=5)

        self.timer = None  # объект Timer
        self.thread = None  # поток для запуска таймера (чтобы не блокировать GUI)
        self.running = False

    def set_timer(self):
        """Создаёт таймер из введённых секунд."""
        try:
            sec = int(self.entry.get())
            if sec < 0:
                raise ValueError
            self.timer = Timer(0, 0, sec)
            self.update_display()
            self.running = False
        except ValueError:
            self.time_label.config(text="Ошибка ввода")

    def update_display(self):
        """Обновляет метку с текущим временем."""
        if self.timer:
            self.time_label.config(text=str(self.timer))

    def start_timer(self):
        """Запускает таймер в отдельном потоке."""
        if not self.timer or self.timer._total_seconds() <= 0:
            return
        if self.running:
            return
        self.running = True
        # Запускаем в потоке, чтобы не блокировать GUI
        self.thread = threading.Thread(target=self._run_timer, daemon=True)
        self.thread.start()

    def _run_timer(self):
        self.timer.start(callback=lambda: self.root.after(0, self.update_display))
        self.running = False

    def stop_timer(self):
        if self.timer and self.running:
            self.timer.stop()
            self.running = False
            self.update_display()

    def reset_timer(self):
        """Сбрасывает таймер."""
        if self.timer:
            self.timer.reset()
            self.running = False


if __name__ == "__main__":
    root = tk.Tk()
    app = TimerGUI(root)
    root.mainloop()
d = Timer(0, 2, 30)
d.start()
